In [6]:
import pandas as pd

NO3 = pd.read_csv('../data_estaciones/BD_NO3.csv')
SO2 = pd.read_csv('../data_estaciones/BD_SO2.csv')

evi_no3 = pd.read_csv('../Etapa 3/EVI_NO3.csv')
evi_so2 = pd.read_csv('../Etapa 3/EVI_SO2.csv')

In [3]:
NO3.head()

,anio,date,CO,NO,NO2,NOX,O3,PM10,PM2.5,PRS,RAINF,RH,SO2,SR,TOUT,WSR,WDR
0,2022,2022-12-01 01:00:00,0.55,2.8,5.8,8.7,29.0,36.0,NaN,726.9,0.0,69.0,2.2,0.0,12.28,9.5,172.0
1,2022,2022-12-01 02:00:00,0.55,3.1,6.9,10.1,28.0,37.0,NaN,726.5,0.0,78.0,2.1,0.0,10.95,8.0,190.0
2,2022,2022-12-01 03:00:00,0.56,2.7,6.6,9.5,28.0,35.0,NaN,726.1,0.0,81.0,2.2,0.0,10.57,4.9,112.0
3,2022,2022-12-01 04:00:00,0.55,2.7,7.0,9.8,27.0,34.0,NaN,725.8,0.0,80.0,2.1,0.0,10.77,4.0,79.0
4,2022,2022-12-01 05:00:00,0.55,2.6,5.8,8.6,27.0,40.0,NaN,725.8,0.0,82.0,2.2,0.0,10.44,4.8,58.0


In [4]:
evi_no3.head()

,estacion,fecha,EVI
0,NO3,2021-01-01,0.0823
1,NO3,2021-01-17,0.0908
2,NO3,2021-02-02,0.0923
3,NO3,2021-02-18,0.0691
4,NO3,2021-03-06,0.1032


In [7]:
# Rangos validos segun SIMA. Los valores fuera de estos limites se reemplazan por NaN.
rangos_validos = {
    'PM10': (0, 999),
    'PM2.5': (0, 999),
    'O3': (0, 185),
    'NO': (0, 500),
    'NO2': (0, 200),
    'SO2': (0, 405),
    'CO': (0, 20),
    'RH': (0, 100),
    'WSR': (0, 75),
    'TOUT': (-6.5, 45.5),
    'SR': (0, 1.26),
    'WDR': (0, 360),
    'RAINF': (0, 80)
}

estaciones = [NO3, SO2]  # Lista de DataFrames de estaciones

for nombre, estacion in zip(['NE2', 'NE3'], estaciones):
    valores_eliminados = 0

    for columna, (minimo, maximo) in rangos_validos.items():
        if columna in estacion.columns:
            valores = pd.to_numeric(estacion[columna], errors='coerce')
            fuera_de_rango = (valores < minimo) | (valores > maximo)
            valores_eliminados += fuera_de_rango.sum()
            estacion.loc[fuera_de_rango, columna] = pd.NA

    print(f'{nombre}: {valores_eliminados} valores fuera de rango reemplazados por NaN')

NE2: 160 valores fuera de rango reemplazados por NaN
NE3: 589 valores fuera de rango reemplazados por NaN


In [10]:
def agregar_mediana_estacion_evi(estacion, evi, fechas_objetivo):
    """Agrega las mediciones horarias al intervalo de cada fecha EVI."""
    estacion = estacion.copy()
    evi = evi.copy()

    estacion['date'] = pd.to_datetime(estacion['date'], errors='coerce')
    evi['fecha'] = pd.to_datetime(evi['fecha'], errors='coerce').dt.normalize()
    estacion = estacion.dropna(subset=['date']).sort_values('date')
    evi = evi.dropna(subset=['fecha']).sort_values('fecha')
    evi = evi[evi['fecha'].isin(fechas_objetivo)].copy()

    columnas_estacion = [
        columna for columna in estacion.columns
        if columna not in ['date', 'anio']
    ]
    for columna in columnas_estacion:
        estacion[columna] = pd.to_numeric(estacion[columna], errors='coerce')

    columnas_numericas = [
        columna for columna in columnas_estacion
        if pd.api.types.is_numeric_dtype(estacion[columna])
    ]

    # Cada medición se asigna a la fecha EVI más reciente anterior o igual.
    mediciones = pd.merge_asof(
        estacion,
        evi[['fecha']],
        left_on='date',
        right_on='fecha',
        direction='backward'
    ).dropna(subset=['fecha'])

    medianas = (
        mediciones.groupby('fecha', as_index=False)[columnas_numericas]
        .median()
    )

    resultado = evi.merge(medianas, on='fecha', how='left')
    # No conservar fechas EVI sin ninguna medición de la estación.
    return (
        resultado
        .dropna(subset=columnas_numericas, how='all')
        .sort_values('fecha')
        .reset_index(drop=True)
    )


# Cada estación conserva sus propias fechas EVI; no se fuerza la misma cobertura temporal.
evi_no3['fecha'] = pd.to_datetime(evi_no3['fecha'], errors='coerce').dt.normalize()
evi_so2['fecha'] = pd.to_datetime(evi_so2['fecha'], errors='coerce').dt.normalize()
fechas_no3 = pd.Index(evi_no3['fecha'].dropna().unique()).sort_values()
fechas_so2 = pd.Index(evi_so2['fecha'].dropna().unique()).sort_values()

NO3_con_EVI = agregar_mediana_estacion_evi(NO3, evi_no3, fechas_no3)
SO2_con_EVI = agregar_mediana_estacion_evi(SO2, evi_so2, fechas_so2)

assert NO3_con_EVI.iloc[:, 3:].notna().any(axis=1).all()
assert SO2_con_EVI.iloc[:, 3:].notna().any(axis=1).all()
print(f'NO3_con_EVI: {NO3_con_EVI.shape[0]} filas')
print(f'SO2_con_EVI: {SO2_con_EVI.shape[0]} filas')
print(f'Diferencia de filas: {abs(len(NO3_con_EVI) - len(SO2_con_EVI))}')

NO3_con_EVI.head()

NO3_con_EVI: 72 filas
SO2_con_EVI: 115 filas
Diferencia de filas: 43


,estacion,fecha,EVI,CO,NO,NO2,NOX,O3,PM10,PM2.5,PRS,RAINF,RH,SO2,SR,TOUT,WSR,WDR
0,NO3,2022-11-17,0.1169,1.10,15.90,23.80,48.30,8.0,63.0,NaN,723.50,0.0,89.5,2.3,0.0,12.420,4.50,148.0
1,NO3,2022-12-03,0.2079,1.08,8.40,19.20,29.20,13.0,69.0,NaN,719.90,0.0,78.0,2.7,0.0,18.880,5.90,126.0
2,NO3,2022-12-19,0.1139,0.80,8.35,20.55,31.45,16.0,70.0,NaN,714.75,0.0,61.0,2.6,0.0,11.945,8.65,99.0
3,NO3,2023-01-01,0.1016,1.02,4.70,18.10,23.50,22.0,83.0,NaN,710.70,0.0,62.0,2.4,0.0,19.430,20.05,341.0
4,NO3,2023-01-17,0.1124,1.30,3.70,13.20,16.60,22.0,70.5,NaN,709.00,0.0,61.0,4.3,0.0,15.790,10.90,53.0


In [11]:
NO3_con_EVI.to_csv('NO3_EVI.csv', index=False)
SO2_con_EVI.to_csv('SO2_EVI.csv', index=False)